In [ ]:
!pip install kaggle

In [ ]:
!ls ~/.kaggle

In [ ]:
!mkdir ~/.kaggle

In [ ]:
!cp ~/Downloads/kaggle.json ~/.kaggle/

In [ ]:
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
! kaggle --version

In [ ]:
!kaggle competitions download -c nlp-getting-started

In [ ]:
!unzip -n nlp-getting-started.zip -d ./nlp_data


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
train  = pd.read_csv('./nlp_data/train.csv')
test = pd.read_csv('./nlp_data/test.csv')

In [ ]:
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
!pip install emoji


In [ ]:
import re
import string
import emoji

In [ ]:
from nltk.tokenize import word_tokenize

In [ ]:
import nltk
nltk.download('stopwords')

In [ ]:
nltk.download('punkt')

In [ ]:
from nltk.stem import WordNetLemmatizer

In [ ]:
nltk.download('wordnet')

In [ ]:
stopwords.words('english')

In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
train.describe()

In [ ]:
test.describe()

In [ ]:
train.info()

In [ ]:
test.info()


In [ ]:
train.isnull().sum(), test.isnull().sum()

In [ ]:
test.shape,train.shape

In [ ]:
train['target'].value_counts()

In [ ]:
train['location'].value_counts()

In [ ]:
train['keyword'].value_counts()


# Tweet Cleaning


In [ ]:
def clean_tweet(tweet):
    # Initialize the lemmatizer
    lemmatizer = WordNetLemmatizer()
    
    # Remove URLs
    tweet = re.sub(r"http\S+|www\S+|https\S+", "", tweet, flags=re.MULTILINE)
    # Remove @ mentions
    tweet = re.sub(r"@\w+", "", tweet)
    # Remove hashtags (keeping the words)
    tweet = re.sub(r"#", "", tweet)
    # Remove punctuation
    tweet = tweet.translate(str.maketrans("", "", string.punctuation))
    # Convert to lowercase
    tweet = tweet.lower()
    # Tokenize the tweet using nltk
    tokens = word_tokenize(tweet)
    # Remove stopwords using nltk's list of stopwords
    tokens = [word for word in tokens if word not in stopwords.words('english')]
    # Lemmatize each word using WordNetLemmatizer
    lemmatized_tokens = [lemmatizer.lemmatize(word) for word in tokens]
    # Join tokens back to a single string
    cleaned_tweet = ' '.join(lemmatized_tokens)
    return cleaned_tweet


In [ ]:
train['cleaned_text'] = train['text'].apply(clean_tweet)

In [ ]:
train[['text','cleaned_text']].head(10)

In [ ]:
train.head()

In [ ]:
train['keyword'] = train['keyword'].fillna('')  
train['location'] = train['location'].fillna('') 
# replacing with empty strings

In [ ]:
train.head()

## Adding a lenght column

In [ ]:
train['tweet_length'] = train['text'].str.len()
test['tweet_length'] = test['text'].str.len()

In [ ]:
train.head()

In [ ]:
test.head()

### Similarly applying cleaning process to test set

In [ ]:
test['cleaned_text'] = test['text'].apply(clean_tweet)

In [ ]:
test[['text','cleaned_text']].head(10)

In [ ]:
test['keyword'] = test['keyword'].fillna('')  
test['location'] = test['location'].fillna('') 
# replacing with empty strings

In [ ]:
test.head()

# Data-plots

### visualize tweet lengths by target value

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(train['tweet_length'], bins=30, alpha=0.7, color='blue', edgecolor='black')
plt.title('Distribution of Tweet Lengths')
plt.xlabel('Tweet Length')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)
plt.show()

In [ ]:

plt.figure(figsize=(5, 3))
plt.scatter(train['tweet_length'], train['target'], alpha=0.5, color='blue')
plt.title('Tweet Length vs Target Value')
plt.xlabel('Tweet Length')
plt.ylabel('Target Value')
plt.yticks([0, 1], ['Non-Disaster', 'Disaster'])  # Customize y-ticks for clarity
plt.grid()
plt.show()


In [ ]:
target_counts = train['target'].value_counts()
target_counts

print("Number of Non-Disaster Tweets:", target_counts[0])
print("Number of Disaster Tweets:", target_counts[1])

### Count the occurrences of keywords

In [ ]:

keywords_count = train['keyword'].value_counts().nlargest(15)

# Plotting the top 10 keywords
plt.figure(figsize=(10, 6))
plt.barh(keywords_count.index, keywords_count.values, color='orange')
plt.title('Top 10 Keyword')
plt.xlabel('Count')
plt.ylabel('Keyword')
plt.gca().invert_yaxis()  # Invert y-axis for better readability
plt.show()


### Count the occurrences of locations

In [ ]:

locations_count = train['location'].value_counts().nlargest(10)

# Plotting the top 10 locations
plt.figure(figsize=(10, 6))
plt.barh(locations_count.index, locations_count.values, color='green')
plt.title('Top 10 Locations')
plt.xlabel('Count')
plt.ylabel('Locations')
plt.gca().invert_yaxis()  # Invert y-axis for better readability
plt.show()


In [ ]:
train.head()

## Model Training

In [ ]:
X = train['cleaned_text']
y = train['target']

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=42)

In [ ]:
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_val_tfidf = tfidf_vectorizer.transform(X_val)

In [ ]:
lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train_tfidf, y_train)

In [ ]:
y_pred = lr_model.predict(X_val_tfidf)


In [ ]:
y_pred

# EVALUATION

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
print("Classification Report:")
print(classification_report(y_val, y_pred))
print(f"Accuracy: {accuracy_score(y_val, y_pred):.4f}")

In [ ]:
correct_predictions = np.sum(y_val == y_pred)
incorrect_predictions = np.sum(y_val != y_pred)
print(f"\nNumber of correct predictions: {correct_predictions}")
print(f"Number of incorrect predictions: {incorrect_predictions}")


In [ ]:
import seaborn as sns
cm = confusion_matrix(y_val, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

In [ ]:
feature_importance = pd.DataFrame({
    'feature': tfidf_vectorizer.get_feature_names_out(),
    'importance': lr_model.coef_[0]
})
feature_importance = feature_importance.sort_values('importance', ascending=False)

print("\nTop 10 most important features:")
print(feature_importance.head(10))

# Test case prediction

In [ ]:
X_test_tfidf = tfidf_vectorizer.transform(test['cleaned_text'])

In [ ]:
test_predictions = lr_model.predict(X_test_tfidf)

In [ ]:
test_predictions

In [ ]:
# final csv
test['target'] = test_predictions
test[['id', 'target']].to_csv('/Users/buddhiprakashmeena/Desktop/ML DAYS/submission.csv', index=False)

In [ ]:
import os

In [ ]:
os.getcwd()